In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b

In [2]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))

x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


In [4]:
# 1. 创建 w1、w2、b，以及包含三个参数的 SGD 优化器
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

# 2. 训练循环
for epoch in range(1, 2001):
    # 设置本轮学习率，并清空旧梯度
    lr = 0.1 / (epoch ** 0.5)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    optimizer.zero_grad()
    
    # 调用 predict，同时使用 x1 和 x2
    y_pred = predict(x1, x2, w1, w2, b)
    
    # 计算逐样本绝对误差和平均损失
    sample_loss = torch.abs(y - y_pred)
    loss = sample_loss.mean()
    
    # 反向传播，更新参数
    loss.backward()
    optimizer.step()

# 3. 在 torch.no_grad() 中，用训练后的参数重新计算预测与损失
with torch.no_grad():
    y_pred = predict(x1, x2, w1, w2, b)
    sample_loss = torch.abs(y - y_pred)
    loss = sample_loss.mean()

# 4. 打印三个参数、x1、x2、y、y_pred、逐样本损失、平均损失及其 shape
for name, val in [('w1', w1), ('w2', w2), ('b', b), 
                  ('x1', x1), ('x2', x2), ('y', y), 
                  ('y_pred', y_pred), ('逐样本损失', sample_loss), ('平均损失', loss)]:
    print(f"{name:8s} shape: {tuple(val.shape)!s:8s} value: {val}")

w1       shape: ()       value: 1.999006986618042
w2       shape: ()       value: -3.0010085105895996
b        shape: ()       value: 0.9993674159049988
x1       shape: (6,)     value: tensor([-2., -1.,  0.,  1.,  2.,  3.])
x2       shape: (6,)     value: tensor([ 1.,  0.,  2., -1.,  3.,  1.])
y        shape: (6,)     value: tensor([-6., -1., -5.,  6., -4.,  4.])
y_pred   shape: (6,)     value: tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954])
逐样本损失    shape: (6,)     value: tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046])
平均损失     shape: ()       value: 0.0023727715015411377


In [ ]:
我的观察与实验
1.现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
答：4；x1 存放了全部 6 个样本在第一个特征维度上的数值
2.对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
答：由上周的非线性/单特征平方公式改为多元线性公式；由上周单一标量参数 变为包含 3 个可学习参数的列表;
上周为单个样本的绝对值损失（shape 为标量 ）； 本周为批量计算——先计算包含 6 个样本的逐样本绝对误差向量，再通过 计算标量平均损失（shape 为 .mean（）)来进行反向传播；
未改变
3.记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
答：1.损失极低：平均损失从最初的较大值平稳下降并最终收敛到了接近 0（0.0024）
2.预测高度拟合：最终的预测向量 与真实标签向量 y_predy（'[-6.， -1.， -5）各项几乎完全吻合。[-6., -1., -5., 6., -4., 4.]
3.参数成功收敛：参数稳定在真实潜在函数关系上
4.为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）
答:梯度大小主要由输入数据 及常数决定，与参数当前的数值大小解耦